In [ ]:
from ipywidgets import widgets
from ipywidgets import interact
style = {'description_width': 'initial'}

In [ ]:
def L_contact(e):
    e = abs(e)
    return max(min(1.0, 3*(0.5 - e)), 0.0)

In [ ]:
def sigma0_rigid(e, xi, V_to_a):
    xi = np.asarray(xi, dtype=float)
    if abs(e) <= 1/6:
        return V_to_a*(1 + 12*e*xi)
    L = L_contact(e)
    if L <= 1e-9:
        return np.zeros_like(xi)
    x0 = np.sign(e)*(3*abs(e) - 1)
    xe = np.sign(e)*0.5
    sigE = 2*V_to_a/L
    sig = sigE*(xi - x0)/(xe - x0)
    return np.maximum(sig, 0.0)

In [ ]:
def plot_kippgrenze_precompute():
    fig, ax = plt.subplots(figsize=(10,6))
    e = np.linspace(0, 0.5, 500)
    Lc = np.array([L_contact(ei) for ei in e])
    ax.axvspan(0, 1/6, color='seagreen', alpha=0.15, label='Volle Aufstandsfläche')
    ax.axvspan(1/6, 1/3, color='gold', alpha=0.20, label='Klaffende Fuge < a/2')
    ax.axvspan(1/3, 0.5, color='darkorange', alpha=0.20, label='Klaffende Fuge > a/2')
    ax.plot(e, Lc, color='black', lw=2.5, label=r'Kontaktbreite $L_c$ / $a$')
    ax.axvline(1/6, color='red', ls=':', label='1. Kernweite ($a/6$)')
    ax.axvline(1/3, color='red', ls='-.', label='2. Kernweite ($a/3$)')
    ax.axvline(0.5, color='darkred', ls='-', lw=2.5, label='EQU-Kippgrenze ($a/2$)')
    ax.set_xlabel(r'$|e_x|$ / $a$')
    ax.set_ylabel(r'$L_c$ / $a$')
    ax.set_xlim(0, 0.51)
    ax.set_ylim(0, 1.05)
    ax.legend(loc='lower left', fontsize=12)
    fig.tight_layout()
    plt.show();

In [ ]:
def draw_zigzag(ax, x, y0, y1, n=6, amp=0.015, **kwargs):
    ys = np.linspace(y0, y1, 2*n+1)
    xs = np.array([x if (i == 0 or i == 2*n) else (x - amp if i % 2 == 1 else x + amp)
                   for i in range(2*n+1)])
    ax.plot(xs, ys, **kwargs)

In [ ]:
def interactive_kippnachweis():
    @interact(V_to_a=widgets.IntSlider(value=100, min=10, max=300, step=10, description=r'$V/a$ / kPa', style=style),
              exi=widgets.FloatSlider(value=0.0, min=-0.55, max=0.55, step=0.01, description=r'$e_x$ / $a$', style=style))
    def _plot(V_to_a, exi):
        e = exi
        L = L_contact(e)
        tipping = L <= 1e-6
        fig, ax = plt.subplots(ncols=2, figsize=(18,8))
        axL, axR = ax

        # --- Schema (links) ---
        axL.axhline(0, color='saddlebrown', lw=1.5, zorder=1)
        axL.plot([-0.75,0.75],[0,0], color='saddlebrown', lw=1.5, zorder=1)
        if not tipping:
            if abs(e) <= 1/6:
                xa, xb = -0.5, 0.5
            else:
                x0 = np.sign(e)*(3*abs(e) - 1)
                xa, xb = (x0, 0.5) if e >= 0 else (-0.5, x0)
            axL.fill_between([xa,xb], -0.05, 0, color='seagreen', alpha=0.6, zorder=2)
            if xa > -0.5 + 1e-6 or xb < 0.5 - 1e-6:
                gx0, gx1 = (-0.5, xa) if xa > -0.5 + 1e-6 else (xb, 0.5)
                axL.fill_between([gx0,gx1], -0.05, 0, facecolor='white', edgecolor='red',
                                  hatch='//', alpha=0.8, zorder=2)
            if abs(e) <= 1/6:
                sigE = V_to_a*(1 + 6*abs(e))
            else:
                sigE = 2*V_to_a/L
        else:
            axL.fill_between([-0.5,0.5], -0.05, 0, facecolor='white', edgecolor='red',
                              hatch='//', alpha=0.8, zorder=2)
            xe = np.sign(e)*0.5 if e != 0 else 0.5
            axL.plot(xe, 0, marker='^', color='darkred', markersize=22, zorder=5)
            sigE = np.inf
        axL.add_patch(plt.Rectangle((-0.5,0), 1.0, 0.12, facecolor='lightgray', edgecolor='black', zorder=3))
        axL.annotate('', xy=(np.clip(e,-0.75,0.75), 0.12), xytext=(np.clip(e,-0.75,0.75), 0.55),
                      arrowprops=dict(arrowstyle='-|>', color='black', lw=3), zorder=6)
        axL.text(np.clip(e,-0.75,0.75), 0.60, r'$V$', ha='center', fontsize=18)
        for val, ls in [(1/6, ':'), (1/3, '-.'), (0.5, '-')]:
            axL.axvline(val, color='red', ls=ls, alpha=0.35)
            axL.axvline(-val, color='red', ls=ls, alpha=0.35)
        axL.set_xlim(-0.75, 0.75)
        axL.set_ylim(-0.12, 0.75)
        axL.set_yticks([])
        axL.set_xlabel(r'$x$ / $a$')

        if tipping:
            title, color = 'KIPPGEFAHR -- EQU-Grenzzustand erreicht/überschritten!', 'darkred'
            info = r'$L_c/a = 0$'
        elif abs(e) > 1/3:
            title, color = 'Klaffende Fuge > a/2 (2. Kernweite überschritten)', 'darkorange'
            info = f'$L_c/a$ = {L:.2f}\n' + r'$\sigma_E$' + f' = {sigE:.0f} kPa'
        elif abs(e) > 1/6:
            title, color = 'Klaffende Fuge (1. Kernweite überschritten)', 'goldenrod'
            info = f'$L_c/a$ = {L:.2f}\n' + r'$\sigma_E$' + f' = {sigE:.0f} kPa'
        else:
            title, color = 'Volle Aufstandsfläche (innerhalb 1. Kernweite)', 'seagreen'
            info = f'$L_c/a$ = {L:.2f}\n' + r'$\sigma_E$' + f' = {sigE:.0f} kPa'
        axL.set_title(title, color=color, fontsize=14)
        axL.text(0.98, 0.95, info, transform=axL.transAxes, ha='right', va='top', fontsize=13,
                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        # --- Sohldruckverteilung (rechts) ---
        if not tipping:
            xi = np.linspace(-0.5, 0.5, 400)
            sig = sigma0_rigid(e, xi, V_to_a)
            axR.plot(xi, sig, color='black', lw=2.5)
            axR.fill_between(xi, sig, 0, color='steelblue', alpha=0.3)
        axR.axvline(e, color='black', ls='--', lw=1.5)
        for val, ls in [(1/6, ':'), (1/3, '-.'), (0.5, '-')]:
            axR.axvline(val, color='red', ls=ls, alpha=0.5)
            axR.axvline(-val, color='red', ls=ls, alpha=0.5)
        axR.set_xlabel(r'$x$ / $a$')
        axR.set_ylabel(r'$\sigma_0$ / kPa')
        axR.set_xlim(-0.5, 0.5)
        axR.set_ylim(bottom=0)
        fig.tight_layout()
        plt.show();

In [ ]:
def spring_contact_solution(e, V_to_a, N, maxiter=50):
    dxi = 1.0/N
    xi = -0.5 + dxi*(np.arange(N) + 0.5)
    active = np.ones(N, dtype=bool)
    c0, c1 = V_to_a, 0.0
    for _ in range(maxiter):
        idx = np.where(active)[0]
        if len(idx) < 2:
            return xi, np.zeros(N), active, False
        xa = xi[idx]
        S0 = len(idx)*dxi
        S1 = np.sum(xa)*dxi
        S2 = np.sum(xa**2)*dxi
        M = np.array([[S0, S1],[S1, S2]])
        rhs = np.array([V_to_a, V_to_a*e])
        try:
            c0, c1 = np.linalg.solve(M, rhs)
        except np.linalg.LinAlgError:
            return xi, np.zeros(N), active, False
        p_all = c0 + c1*xi
        new_active = p_all >= -1e-9*max(1.0, abs(V_to_a))
        if np.array_equal(new_active, active):
            p = np.where(active, np.maximum(c0 + c1*xi, 0.0), 0.0)
            return xi, p, active, True
        active = new_active
    p = np.where(active, np.maximum(c0 + c1*xi, 0.0), 0.0)
    return xi, p, active, False

In [ ]:
def interactive_federmodell():
    @interact(V_to_a=widgets.IntSlider(value=100, min=10, max=300, step=10, description=r'$V/a$ / kPa', style=style),
              exi=widgets.FloatSlider(value=0.0, min=-0.55, max=0.55, step=0.01, description=r'$e_x$ / $a$', style=style),
              N=widgets.IntSlider(value=15, min=5, max=41, step=2, description=r'Anzahl Federn $N$', style=style))
    def _plot(V_to_a, exi, N):
        xi, p, active, feasible = spring_contact_solution(exi, V_to_a, N)
        dxi = 1.0/N
        fig, ax = plt.subplots(ncols=2, figsize=(18,8))
        axL, axR = ax

        axL.add_patch(plt.Rectangle((-0.5,0.30), 1.0, 0.08, facecolor='lightgray', edgecolor='black', zorder=5))
        axL.axhline(0, color='saddlebrown', lw=1.5, zorder=1)
        axL.plot([-0.75,0.75],[0,0], color='saddlebrown', lw=1.5, zorder=1)
        pmax = max(p.max(), 1e-9)
        for xi_i, p_i, act_i in zip(xi, p, active):
            if act_i and p_i > 1e-9*pmax:
                lw = 1.0 + 3.0*p_i/pmax
                draw_zigzag(axL, xi_i, 0, 0.30, n=5, amp=0.012, color='seagreen', lw=lw, zorder=4)
                axL.plot(xi_i, 0, marker='s', color='seagreen', markersize=4, zorder=4)
            else:
                draw_zigzag(axL, xi_i, 0, 0.20, n=4, amp=0.010, color='lightgray', lw=1.2, ls=':', zorder=2)
        axL.annotate('', xy=(np.clip(exi,-0.75,0.75), 0.38), xytext=(np.clip(exi,-0.75,0.75), 0.75),
                      arrowprops=dict(arrowstyle='-|>', color='black', lw=3), zorder=6)
        axL.text(np.clip(exi,-0.75,0.75), 0.80, r'$V$', ha='center', fontsize=18)
        for val, ls in [(1/6, ':'), (1/3, '-.'), (0.5, '-')]:
            axL.axvline(val, color='red', ls=ls, alpha=0.3)
            axL.axvline(-val, color='red', ls=ls, alpha=0.3)
        axL.set_xlim(-0.75, 0.75)
        axL.set_ylim(-0.05, 0.9)
        axL.set_yticks([])
        axL.set_xlabel(r'$x$ / $a$')

        n_active = int(active.sum())
        if feasible:
            title = f'{n_active} / {N} Federn im Kontakt (Druck), Kontakt ~ {n_active/N*100:.0f}%'
            color = 'seagreen' if n_active == N else ('goldenrod' if n_active > N//3 else 'darkorange')
        else:
            title = 'KEIN GLEICHGEWICHT -- Fundament kippt (zu wenige Federn im Kontakt)'
            color = 'darkred'
        axL.set_title(title, color=color, fontsize=14)

        axR.bar(xi, p, width=dxi*0.9, color='seagreen', alpha=0.6, label='Federkraft (diskret)', zorder=3)
        xc = np.linspace(-0.5, 0.5, 400)
        if abs(exi) < 0.5:
            axR.plot(xc, sigma0_rigid(exi, xc, V_to_a), color='black', lw=2, label='Kontinuierliche Lösung', zorder=4)
        for val, ls in [(1/6, ':'), (1/3, '-.'), (0.5, '-')]:
            axR.axvline(val, color='red', ls=ls, alpha=0.4)
            axR.axvline(-val, color='red', ls=ls, alpha=0.4)
        axR.set_xlabel(r'$x$ / $a$')
        axR.set_ylabel(r'$\sigma_0$ / kPa')
        axR.set_xlim(-0.5, 0.5)
        axR.set_ylim(bottom=0)
        axR.legend(fontsize=12)
        fig.tight_layout()
        plt.show();

In [ ]:
def theta_crit_rigid(a, zG0):
    return np.arctan(a/(2*zG0))

In [ ]:
def plot_schwerpunktlage(a=4.0, H=20.0):
    zG0 = H/2
    theta_crit = theta_crit_rigid(a, zG0)
    theta_max = min(2.2*theta_crit, 0.5*np.pi)
    theta = np.linspace(0.0, theta_max, 500)

    # Starre Rotation um die belastete Sohlkante:
    # globale Schwerpunktordinate relativ zur Ausgangslage.
    dz_rigid = zG0*(np.cos(theta) - 1.0) + 0.5*a*np.sin(theta)
    # Kinematischer Grenzfall ohne Anhebung der oberen Sohlkante:
    # Gegenüber der Kantenrotation sinkt der Körper um a*sin(theta) ab.
    dz_no_edge_lift = zG0*(np.cos(theta) - 1.0) - 0.5*a*np.sin(theta)
    dz_crit = zG0*(np.cos(theta_crit) - 1.0) + 0.5*a*np.sin(theta_crit)

    fig, ax = plt.subplots(figsize=(10, 6.5))
    theta_deg = np.degrees(theta)
    ax.plot(theta_deg, dz_rigid/a, color='black', lw=2.5,
            label='starre Rotation um die Sohlkante')
    ax.plot(theta_deg, dz_no_edge_lift/a, color='steelblue', lw=1.8, ls='--',
            label='ohne Kantenhebung (kinematischer Grenzfall)')
    ax.fill_between(theta_deg, dz_no_edge_lift/a, dz_rigid/a,
                    color='steelblue', alpha=0.10,
                    label='Bereich bei verminderter Kantenhebung')
    ax.axhline(0.0, color='grey', lw=1.0)
    ax.axvline(np.degrees(theta_crit), color='darkred', lw=2.0, ls='--',
               label=r'$\theta_{\mathrm{krit,starr}}$')
    ax.plot(np.degrees(theta_crit), dz_crit/a, marker='o', color='darkred', ms=8)
    ax.annotate('maximale Schwerpunktlage',
                xy=(np.degrees(theta_crit), dz_crit/a),
                xytext=(10, -24), textcoords='offset points', fontsize=11,
                va='top')
    ax.set_xlabel(r'Neigung $\theta$ / deg')
    ax.set_ylabel(r'$(z_\mathrm{G}(\theta)-z_\mathrm{G0})/a$')
    ax.set_title('Änderung der globalen Schwerpunktlage beim Kippen')
    ax.set_xlim(0.0, np.degrees(theta_max))
    ax.legend(fontsize=11)
    fig.tight_layout()
    plt.show();

In [ ]:
def sigma_spring_nl(s, ks, sref, n):
    s = np.asarray(s, dtype=float)
    spos = np.maximum(s, 0.0)
    sig = ks*sref*(spos/sref)**n
    return np.where(s > 0, sig, 0.0)

In [ ]:
def solve_s0_nl_vec(thetas, V, a, N, ks, n, sref, maxiter=60):
    thetas = np.atleast_1d(np.asarray(thetas, dtype=float))
    dx = a/N
    xi = -a/2 + dx*(np.arange(N) + 0.5)
    XI = xi[None, :]
    ST = np.sin(thetas)[:, None]
    CT = np.cos(thetas)[:, None]
    lo = np.full(thetas.shape, -1e3*sref)  # s0 darf negativ werden (Kontaktzone ganz am Rand)
    hi = np.full(thetas.shape, 1e6*sref)
    for _ in range(maxiter):
        mid = 0.5*(lo + hi)
        s = mid[:, None] + ST*XI                 # exakt: xi*sin(theta), keine Kleinwinkelnäherung
        resid = np.sum(sigma_spring_nl(s, ks, sref, n), axis=1)*dx - V
        hi = np.where(resid > 0, mid, hi)
        lo = np.where(resid > 0, lo, mid)
    s0 = 0.5*(lo + hi)
    s_final = s0[:, None] + ST*XI
    sig = sigma_spring_nl(s_final, ks, sref, n)
    M = np.sum(sig*XI*CT, axis=1)*dx              # exakter Hebelarm: xi*cos(theta), keine Näherung cos(theta)=1
    return s0, M, xi

In [ ]:
def M_dest(theta, V, zG0):
    return V*zG0*np.sin(theta)

In [ ]:
def M_from_u(s0, umin, umax, st, ct, ks, sref, n):
    return (ct/st**2) * ks*sref**(1-n) * ( (umax**(n+2)-umin**(n+2))/(n+2) - s0*(umax**(n+1)-umin**(n+1))/(n+1) )

def solve_theta_analytic(thetas, V, a, ks, n, sref, maxiter=80):
    thetas = np.atleast_1d(np.asarray(thetas, dtype=float))
    st, ct = np.sin(thetas), np.cos(thetas)
    zero_mask = np.abs(st) < 1e-12
    st_safe = np.where(zero_mask, 1.0, st)

    # Fall 1: klaffende Fuge (Kontaktzone [x0,a/2], geschlossene Form über s_edge)
    s_edge = (V*(n+1)*st_safe*sref**(n-1)/ks)**(1/(n+1))
    s0_partial = s_edge - st_safe*a/2
    M_partial = M_from_u(s0_partial, 0.0, s_edge, st_safe, ct, ks, sref, n)

    # Fall 2: volle Aufstandsfläche (Bisektion in s0, keine räumliche Diskretisierung)
    lo = st_safe*a/2
    hi = st_safe*a/2 + 1e6*sref
    for _ in range(maxiter):
        mid = 0.5*(lo + hi)
        umin, umax = mid - st_safe*a/2, mid + st_safe*a/2
        Vm = ks*sref**(1-n)/st_safe * (umax**(n+1) - umin**(n+1))/(n+1)
        hi = np.where(Vm > V, mid, hi)
        lo = np.where(Vm > V, lo, mid)
    s0_full = 0.5*(lo + hi)
    umin_full, umax_full = s0_full - st_safe*a/2, s0_full + st_safe*a/2
    M_full = M_from_u(s0_full, umin_full, umax_full, st_safe, ct, ks, sref, n)

    V_boundary = ks*sref**(1-n)*(st_safe*a)**(n+1)/(st_safe*(n+1))
    full_mask = V >= V_boundary
    s0 = np.where(full_mask, s0_full, s0_partial)
    M = np.where(full_mask, M_full, M_partial)
    s0 = np.where(zero_mask, sref, s0)
    M = np.where(zero_mask, 0.0, M)
    return s0, M

In [ ]:
def ks_nominal_and_sref(ks, n, V, a):
    ks_nom = ks/n
    sref = (V/a)/ks_nom
    return ks_nom, sref

In [ ]:
def find_theta_star(V, a, zG0, ks, n, npts=2000, theta_min=1e-4):
    ks_crit = 12*V*zG0/a**3
    if ks <= ks_crit:
        return 0.0
    theta_rigid = theta_crit_rigid(a, zG0)
    ks_nom, sref = ks_nominal_and_sref(ks, n, V, a)
    thetas = np.linspace(theta_min, theta_rigid*(1 - 1e-9), npts)
    _, Mf = solve_theta_analytic(thetas, V, a, ks_nom, n, sref)
    g = Mf - M_dest(thetas, V, zG0)
    if g[0] <= 0:
        return 0.0
    idx = np.where(g < 0)[0]
    if len(idx) == 0:
        return theta_rigid
    i1 = idx[0]
    th0, th1 = thetas[i1-1], thetas[i1]
    g0, g1 = g[i1-1], g[i1]
    return th0 - g0*(th1 - th0)/(g1 - g0)

In [ ]:
def rot(pts, ang):
    c, s = np.cos(ang), np.sin(ang)
    R = np.array([[c, -s], [s, c]])
    return pts @ R.T

In [ ]:
def interactive_turm_kippen():
    @interact(a=widgets.FloatSlider(value=4.0, min=2.0, max=8.0, step=0.5, description=r'Turmbreite $a$ / m', style=style),
              H=widgets.FloatSlider(value=20.0, min=8.0, max=40.0, step=1.0, description=r'Turmhöhe $H$ / m', style=style),
              k_ratio=widgets.FloatSlider(value=1.5, min=0.2, max=6.0, step=0.1, description=r'$k_s\,/\,k_{s,krit}$', style=style),
              n=widgets.FloatSlider(value=1.0, min=0.4, max=2.5, step=0.1, description=r'Nichtlinearität $n$', style=style),
              theta_deg=widgets.FloatSlider(value=2.0, min=0.0, max=20.0, step=0.2, description=r'Neigung $\theta$ / deg', style=style))
    def _plot(a, H, k_ratio, n, theta_deg):
        gamma = 25.0
        zG0 = H/2
        V = gamma*a*H  # Vertikallinienlast in kN/m
        N_vis = 25    # Anzahl gezeichneter Federn im Schema (nur für die Illustration)
        ks_crit = 12*V*zG0/a**3
        ks = k_ratio*ks_crit
        theta_rigid = theta_crit_rigid(a, zG0)
        theta = np.radians(theta_deg)

        ks_nom, sref = ks_nominal_and_sref(ks, n, V, a)
        s0, _, xi = solve_s0_nl_vec(np.array([theta]), V, a, N_vis, ks_nom, n, sref)
        s0 = s0[0]
        sig = sigma_spring_nl(s0 + np.sin(theta)*xi, ks_nom, sref, n)

        _, Mf_now = solve_theta_analytic(np.array([theta]), V, a, ks_nom, n, sref)
        Mf_now = Mf_now[0]
        Md_now = M_dest(theta, V, zG0)
        theta_star = find_theta_star(V, a, zG0, ks, n)

        fig, ax = plt.subplots(ncols=2, figsize=(18,9))
        axL, axR = ax

        corners = np.array([[-a/2,0],[a/2,0],[a/2,H],[-a/2,H]])
        corners_r = rot(corners, -theta)
        base_r = rot(np.array([[-a/2,0],[a/2,0]]), -theta)
        ground_y = min(-1.0, base_r[:,1].min() - 0.5)
        axL.add_patch(plt.Polygon(corners_r, closed=True, facecolor='lightgray', edgecolor='black', zorder=5))
        cg_r = rot(np.array([[0, H/2]]), -theta)[0]
        axL.plot([cg_r[0], cg_r[0]], [cg_r[1], ground_y], color='black', ls='--', lw=1.5, zorder=6)
        axL.plot(*cg_r, marker='o', color='black', zorder=7)
        axL.text(cg_r[0], cg_r[1]+0.5, 'CG', ha='center', fontsize=11)

        axL.plot(base_r[:,0], base_r[:,1], color='black', lw=3, zorder=4)
        axL.axhline(ground_y, color='saddlebrown', lw=1.5, zorder=1)

        # Erste und zweite Kernweite entlang der geneigten Sohlfuge.
        normal_r = rot(np.array([[0.0, 1.0]]), -theta)[0]
        tick_len = 0.12*a
        for frac, kw_color, kw_label in [(1/6, 'crimson', '1. KW ($a/6$)'),
                                          (1/3, 'darkorange', '2. KW ($a/3$)')]:
            kw_pts = rot(np.array([[-frac*a, 0.0], [frac*a, 0.0]]), -theta)
            for j, (px, py) in enumerate(kw_pts):
                p0 = np.array([px, py]) - 0.5*tick_len*normal_r
                p1 = np.array([px, py]) + 0.5*tick_len*normal_r
                axL.plot([p0[0], p1[0]], [p0[1], p1[1]], color=kw_color,
                         lw=2.2, zorder=8,
                         label=kw_label if j == 0 else '_nolegend_')
        axL.legend(loc='upper left', fontsize=10, framealpha=0.9)

        pmax = max(sig.max(), 1e-9)
        base_pts = rot(np.column_stack([xi, np.zeros_like(xi)]), -theta)
        for (bx,by), p_i in zip(base_pts, sig):
            if p_i > 1e-9*pmax:
                lw = 1.0 + 3.0*p_i/pmax
                draw_zigzag(axL, bx, ground_y, by, n=5, amp=0.15, color='seagreen', lw=lw, zorder=3)
            else:
                draw_zigzag(axL, bx, ground_y, by, n=4, amp=0.12, color='lightgray', lw=1.0, ls=':', zorder=2)

        axL.set_xlim(-H*0.6, H*0.6)
        axL.set_ylim(ground_y-0.5, H*1.15)
        axL.set_aspect('equal')
        axL.set_xlabel('$x$ / m')
        G_now = Mf_now - Md_now
        if abs(theta) < 1e-12:
            if ks > ks_crit:
                state, color = 'lokal stabil', 'seagreen'
            elif ks < ks_crit:
                state, color = 'lokal instabil', 'darkred'
            else:
                state, color = 'linearisiert kritisch', 'goldenrod'
        elif theta >= theta_rigid:
            state, color = 'jenseits der geometrischen Kippgrenze', 'darkred'
        elif G_now > 0:
            state, color = 'rückstellendes Moment', 'seagreen'
        else:
            state, color = 'kippendes Moment', 'darkred'
        axL.set_title(f'$\\theta$ = {theta_deg:.1f}$°$ ({state})', color=color, fontsize=14)

        theta_plot_max = max(theta_rigid*1.02, theta*1.02, np.radians(0.2))
        thetas = theta_plot_max*np.linspace(0.0, 1.0, 400)**2  # dichter nahe theta=0
        _, Mf_curve = solve_theta_analytic(thetas, V, a, ks_nom, n, sref)
        Md_curve = M_dest(thetas, V, zG0)
        axR.plot(np.degrees(thetas), Md_curve, color='black', lw=2, label=r'$M_\mathrm{dest}=V z_{\mathrm{G0}} \sin\theta$')
        axR.plot(np.degrees(thetas), Mf_curve, color='seagreen', lw=2, label=r'$M_\mathrm{found}(\theta)$')
        axR.axvline(np.degrees(theta_rigid), color='darkred', ls='-', lw=2, label=r'$\theta_{krit,starr}$')
        if theta_star > 1e-6:
            axR.axvline(np.degrees(theta_star), color='orange', ls='--', lw=2, label=r'$\theta^*$ (Federmodell)')
        axR.plot(theta_deg, Md_now, marker='o', color=color, markersize=10, zorder=5)
        axR.set_xlabel(r'$\theta$ / deg')
        axR.set_ylabel('Moment / kNm/m')
        axR.set_xlim(0, np.degrees(theta_plot_max))
        axR.set_ylim(bottom=0)
        axR.legend(fontsize=11)
        fig.tight_layout()
        plt.show();

In [ ]:
def find_theta_equilibrium(V, a, zG0, H_tip, Fh, ks, n, npts=4000, theta_min=1e-5):
    theta_rigid = theta_crit_rigid(a, zG0)
    if Fh <= 0:
        return 0.0
    ks_nom, sref = ks_nominal_and_sref(ks, n, V, a)
    thetas = np.linspace(theta_min, theta_rigid*(1 - 1e-9), npts)
    _, Mf = solve_theta_analytic(thetas, V, a, ks_nom, n, sref)
    G = Mf - M_dest(thetas, V, zG0) - Fh*H_tip*np.cos(thetas)
    idx = np.where(G >= 0)[0]
    if len(idx) == 0:
        return None
    i1 = idx[0]
    if i1 == 0:
        return thetas[0]
    th0, th1 = thetas[i1-1], thetas[i1]
    g0, g1 = G[i1-1], G[i1]
    return th0 - g0*(th1 - th0)/(g1 - g0)

In [ ]:
def interactive_turm_gleichgewicht():
    @interact(a=widgets.FloatSlider(value=4.0, min=2.0, max=8.0, step=0.5, description=r'Turmbreite $a$ / m', style=style),
              H=widgets.FloatSlider(value=20.0, min=8.0, max=40.0, step=1.0, description=r'Turmhöhe $H$ / m', style=style),
              k_ratio=widgets.FloatSlider(value=1.5, min=0.2, max=6.0, step=0.1, description=r'$k_s\,/\,k_{s,krit}$', style=style),
              n=widgets.FloatSlider(value=1.0, min=0.4, max=2.5, step=0.1, description=r'Nichtlinearität $n$', style=style),
              eta=widgets.FloatSlider(value=0.006, min=0.0, max=0.05, step=0.001, description=r'$F_h\,/\,V$', style=style, readout_format='.3f'))
    def _plot(a, H, k_ratio, n, eta):
        gamma = 25.0
        zG0 = H/2
        V = gamma*a*H  # Vertikallinienlast in kN/m
        Fh = eta*V     # Horizontallinienlast an der Turmspitze in kN/m
        N_vis = 25
        ks_crit = 12*V*zG0/a**3
        ks = k_ratio*ks_crit
        theta_rigid = theta_crit_rigid(a, zG0)
        ks_nom, sref = ks_nominal_and_sref(ks, n, V, a)

        fig, (axA, axB) = plt.subplots(ncols=2, figsize=(18,9))

        # ---------------------------------------------------------------
        # Panel A: starrer, aufrecht bleibender Turm (Ersatzausmitte e_x)
        # ---------------------------------------------------------------
        e_xA = Fh*H/V
        e_nA = e_xA/a
        xiA, pA, activeA, feasibleA = spring_contact_solution(e_nA, V/a, N_vis)
        xA = xiA*a

        axA.add_patch(plt.Rectangle((-a/2, 0), a, H, facecolor='lightgray', edgecolor='black', zorder=5))
        axA.plot([-a/2, a/2], [0, 0], color='black', lw=3, zorder=4)
        ground_yA = -1.0
        axA.axhline(ground_yA, color='saddlebrown', lw=1.5, zorder=1)

        pmaxA = max(pA.max(), 1e-9)
        for x_i, p_i, act_i in zip(xA, pA, activeA):
            if act_i and p_i > 1e-9*pmaxA:
                lw = 1.0 + 3.0*p_i/pmaxA
                draw_zigzag(axA, x_i, ground_yA, 0, n=5, amp=0.15, color='seagreen', lw=lw, zorder=3)
            else:
                draw_zigzag(axA, x_i, ground_yA, 0, n=4, amp=0.12, color='lightgray', lw=1.0, ls=':', zorder=2)

        axA.annotate('', xy=(0, H), xytext=(0, H*1.12),
                     arrowprops=dict(arrowstyle='-|>', color='black', lw=3), zorder=6)
        axA.text(0, H*1.16, r'$V$', ha='center', fontsize=16)
        axA.annotate('', xy=(a*0.9, H), xytext=(0, H),
                     arrowprops=dict(arrowstyle='-|>', color='darkblue', lw=3), zorder=6)
        axA.text(a*0.95, H, r'$F_h$', ha='left', va='center', fontsize=16, color='darkblue')

        axA.plot([e_xA, e_xA], [ground_yA, H*1.05], color='black', ls='--', lw=1.5, zorder=6)
        axA.text(e_xA, H*1.08, r'$e_x$', ha='center', fontsize=12)

        for frac, kw_color, kw_label in [(1/6, 'crimson', '1. KW ($a/6$)'), (1/3, 'darkorange', '2. KW ($a/3$)')]:
            for j, sgn in enumerate([-1, 1]):
                axA.plot([sgn*frac*a, sgn*frac*a], [-0.15, 0.15], color=kw_color, lw=2.2, zorder=8,
                         label=kw_label if j == 0 else '_nolegend_')
        axA.legend(loc='upper left', fontsize=9, framealpha=0.9)
        axA.text(0.98, 0.95, f'$e_x/a$ = {e_nA:.3f}', transform=axA.transAxes, ha='right', va='top', fontsize=12,
                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        n_activeA = int(activeA.sum())
        if not feasibleA:
            titleA, colorA = 'KEIN GLEICHGEWICHT -- kippt (starre Idealisierung)', 'darkred'
        elif abs(e_nA) <= 1/6:
            titleA, colorA = 'volle Aufstandsfläche', 'seagreen'
        elif abs(e_nA) <= 1/3:
            titleA, colorA = 'klaffende Fuge < a/2', 'goldenrod'
        else:
            titleA, colorA = 'klaffende Fuge > a/2', 'darkorange'
        axA.set_title(f'Starr, aufrecht: {n_activeA}/{N_vis} Federn aktiv -- {titleA}', color=colorA, fontsize=13)
        axA.set_xlim(-H*0.6, H*0.6)
        axA.set_ylim(ground_yA - 0.5, H*1.25)
        axA.set_aspect('equal')
        axA.set_xlabel('$x$ / m')

        # ---------------------------------------------------------------
        # Panel B: elastische Neigungsantwort (volles Momentengleichgewicht)
        # ---------------------------------------------------------------
        theta_eq = find_theta_equilibrium(V, a, zG0, H, Fh, ks, n)
        tippingB = theta_eq is None
        theta_draw = theta_rigid if tippingB else theta_eq

        s0, _, xiB = solve_s0_nl_vec(np.array([theta_draw]), V, a, N_vis, ks_nom, n, sref)
        s0 = s0[0]
        sigB = sigma_spring_nl(s0 + np.sin(theta_draw)*xiB, ks_nom, sref, n)
        Mfound_draw = np.sum(sigB*xiB*np.cos(theta_draw))*(a/N_vis)
        e_xB = Mfound_draw/(V*np.cos(theta_draw))

        corners = np.array([[-a/2,0],[a/2,0],[a/2,H],[-a/2,H]])
        corners_r = rot(corners, -theta_draw)
        base_r = rot(np.array([[-a/2,0],[a/2,0]]), -theta_draw)
        ground_yB = min(-1.0, base_r[:,1].min() - 0.5)
        axB.add_patch(plt.Polygon(corners_r, closed=True, facecolor='lightgray', edgecolor='black', zorder=5))
        cg_r = rot(np.array([[0, H/2]]), -theta_draw)[0]
        tip_r = rot(np.array([[0, H]]), -theta_draw)[0]
        axB.plot([cg_r[0], cg_r[0]], [cg_r[1], ground_yB], color='black', ls='--', lw=1.5, zorder=6)
        axB.plot(*cg_r, marker='o', color='black', zorder=7)
        axB.text(cg_r[0], cg_r[1]+0.5, 'CG', ha='center', fontsize=11)

        axB.plot(base_r[:,0], base_r[:,1], color='black', lw=3, zorder=4)
        axB.axhline(ground_yB, color='saddlebrown', lw=1.5, zorder=1)

        # Lasten: V senkrecht durch den Schwerpunkt, F_h waagerecht an der Turmspitze.
        axB.annotate('', xy=(cg_r[0], cg_r[1]), xytext=(cg_r[0], cg_r[1]+3.0),
                      arrowprops=dict(arrowstyle='-|>', color='black', lw=3), zorder=6)
        axB.text(cg_r[0], cg_r[1]+3.4, r'$V$', ha='center', fontsize=14)
        Fh_tip_len = max(0.15*H, 2.0)
        axB.annotate('', xy=(tip_r[0]+Fh_tip_len, tip_r[1]), xytext=(tip_r[0], tip_r[1]),
                      arrowprops=dict(arrowstyle='-|>', color='darkblue', lw=3), zorder=6)
        axB.text(tip_r[0]+Fh_tip_len*1.1, tip_r[1], r'$F_h$', ha='left', va='center', fontsize=14, color='darkblue')

        normal_r = rot(np.array([[0.0, 1.0]]), -theta_draw)[0]
        tick_len = 0.12*a
        for frac, kw_color, kw_label in [(1/6, 'crimson', '1. KW ($a/6$)'), (1/3, 'darkorange', '2. KW ($a/3$)')]:
            kw_pts = rot(np.array([[-frac*a, 0.0], [frac*a, 0.0]]), -theta_draw)
            for j, (px, py) in enumerate(kw_pts):
                p0 = np.array([px, py]) - 0.5*tick_len*normal_r
                p1 = np.array([px, py]) + 0.5*tick_len*normal_r
                axB.plot([p0[0], p1[0]], [p0[1], p1[1]], color=kw_color, lw=2.2, zorder=8,
                         label=kw_label if j == 0 else '_nolegend_')
        # Ersatzausmitte des Gleichgewichtszustands, körperfest entlang der Sohlfuge.
        if not tippingB:
            wl_pt = rot(np.array([[e_xB, 0.0]]), -theta_draw)[0]
            axB.plot(wl_pt[0], wl_pt[1], marker='D', color='black', ms=9, zorder=9, label=r'$e_x$ (Gleichgewicht)')
        axB.legend(loc='upper left', fontsize=9, framealpha=0.9)
        info_txt = f'$e_x/a$ = {e_xB/a:.3f}' if not tippingB else 'kein GG'
        axB.text(0.98, 0.95, info_txt, transform=axB.transAxes, ha='right', va='top', fontsize=12,
                  bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        pmaxB = max(sigB.max(), 1e-9)
        base_pts = rot(np.column_stack([xiB, np.zeros_like(xiB)]), -theta_draw)
        for (bx,by), p_i in zip(base_pts, sigB):
            if p_i > 1e-9*pmaxB:
                lw = 1.0 + 3.0*p_i/pmaxB
                draw_zigzag(axB, bx, ground_yB, by, n=5, amp=0.15, color='seagreen', lw=lw, zorder=3)
            else:
                draw_zigzag(axB, bx, ground_yB, by, n=4, amp=0.12, color='lightgray', lw=1.0, ls=':', zorder=2)

        axB.set_xlim(-H*0.6, H*0.6)
        axB.set_ylim(ground_yB - 0.5, H*1.25)
        axB.set_aspect('equal')
        axB.set_xlabel('$x$ / m')
        n_activeB = int(np.sum(sigB > 1e-9*pmaxB))
        if tippingB:
            titleB, colorB = 'KEIN GLEICHGEWICHT -- Turm kippt', 'darkred'
        else:
            titleB, colorB = f'$\\theta_\\mathrm{{eq}}$ = {np.degrees(theta_draw):.2f}$°$, {n_activeB}/{N_vis} Federn aktiv', 'seagreen'
        axB.set_title(f'Elastische Bettung: {titleB}', color=colorB, fontsize=13)

        fig.tight_layout()
        plt.show();

In [ ]:
def plot_kippschwelle_federsteifigkeit(a=4.0, H=20.0):
    gamma = 25.0
    zG0 = H/2
    V = gamma*a*H  # Vertikallinienlast in kN/m
    ks_crit = 12*V*zG0/a**3
    theta_rigid = theta_crit_rigid(a, zG0)
    k_base = np.logspace(np.log10(0.3), 4, 180)
    # Nahe k_s/k_s,krit = 1 gilt theta* ~ sqrt(k_s/k_s,krit - 1) (unendliche Steigung am
    # Einsatzpunkt); eine in u=sqrt(k_ratio-1) gleichmäßige Verdichtung löst das auflösungsfrei auf.
    u = np.linspace(1e-4, 1.5, 250)
    k_near = 1.0 + u**2
    k_ratios = np.unique(np.concatenate((k_base, [1.0], k_near)))
    fig, ax = plt.subplots(figsize=(11,6.5))
    for n, label, color in [(0.6, 'degressiv ($n=0.6$)', 'darkorange'),
                             (1.0, 'linear ($n=1.0$)', 'black'),
                             (1.8, 'progressiv ($n=1.8$)', 'seagreen')]:
        thetas_star = np.array([find_theta_star(V, a, zG0, kr*ks_crit, n, npts=3000) for kr in k_ratios])
        ax.plot(k_ratios, np.degrees(thetas_star), color=color, lw=2.5, label=label)
    ax.axhline(np.degrees(theta_rigid), color='darkred', ls='-', lw=2, label=r'geom. Grenze $\theta_{krit,starr}$')
    ax.axvline(1.0, color='grey', ls=':', lw=1.5)
    ax.set_xscale('log')
    ax.set_xlabel(r'$k_s\,/\,k_{s,krit}$ (linearer Bezugswert, log. Skala)')
    ax.set_ylabel(r'instabile Winkelschwelle $\theta^*$ / deg')
    ax.set_xlim(k_base[0], k_base[-1])
    ax.set_ylim(0, np.degrees(theta_rigid)*1.05)
    ax.legend(fontsize=12)
    ax.set_title(f'$a$={a:.1f} m, $H$={H:.1f} m  ($k_{{s,krit}}$={ks_crit:.0f} kN/m³)', fontsize=13)
    fig.tight_layout()
    plt.show();

In [ ]:
from scipy.integrate import cumulative_trapezoid
from matplotlib import cm

In [ ]:
def Pi_total(a, H, ks, n, npts=1500):
    gamma = 25.0
    zG0 = H/2
    V = gamma*a*H  # Vertikallinienlast in kN/m
    ks_nom, sref = ks_nominal_and_sref(ks, n, V, a)
    theta_rigid = theta_crit_rigid(a, zG0)
    thetas = np.linspace(0.0, theta_rigid*0.999, npts)
    s0, Mf = solve_theta_analytic(thetas, V, a, ks_nom, n, sref)
    G = Mf - M_dest(thetas, V, zG0)
    Pi_red = cumulative_trapezoid(G, thetas, initial=0.0)
    return thetas, Pi_red, s0

In [ ]:
def plot_energy_landscape(a=4.0, H=20.0):
    zG0 = H/2
    V = 25.0*a*H
    ks_crit = 12*V*zG0/a**3
    fig, ax = plt.subplots(figsize=(10,6.5))
    ax2 = ax.twinx()
    for kr, color in zip([0.7, 1.0, 1.5, 3.0], ['darkred', 'goldenrod', 'seagreen', 'black']):
        thetas, Pi, s0 = Pi_total(a, H, kr*ks_crit, 1.0)
        ax.plot(np.degrees(thetas), Pi, color=color, lw=2.5, label=fr'$k_s/k_{{s,krit}}={kr}$')
        zG_global = zG0*np.cos(thetas) - s0
        zG_over_initial = zG_global/zG_global[0]
        ax2.plot(np.degrees(thetas), zG_over_initial, color=color, lw=1.3, ls='--', alpha=0.6)
    ax.axvline(0, color='grey', ls=':')
    ax.set_xlabel(r'$\theta$ / deg')
    ax.set_ylabel(r'$\Pi(\theta)$ / kNm/m')
    ax.set_title('Energielandschaft (n=1, linear)', fontsize=13)
    ax2.set_ylabel(r'$z_\mathrm{G}(\theta)\,/\,z_\mathrm{G}(0)$', color='dimgrey')
    ax2.tick_params(axis='y', colors='dimgrey')

    from matplotlib.lines import Line2D
    lines1, labels1 = ax.get_legend_handles_labels()
    lines1.append(Line2D([0], [0], color='grey', lw=1.3, ls='--'))
    labels1.append(r'Schwerpunktlage $z_\mathrm{G}(\theta)/z_\mathrm{G}(0)$ (je $k_s$, gestrichelt)')
    ax.legend(lines1, labels1, loc='lower left', fontsize=10)
    fig.tight_layout()
    plt.show();

In [ ]:
def hollow_foundation_properties(a, c):
    if not 0 <= c < a:
        raise ValueError('Es muss 0 <= c < a gelten.')
    r = c/a
    contact_width = a - c
    I_contact = (a**3 - c**3)/12
    e1 = a*(1 + r + r**2)/6
    e2 = a*(1 + r + r**2)/(3*(1 + r))
    return contact_width, I_contact, e1, e2

def _hollow_active_integrals(x0, a, c):
    x0 = np.asarray(x0, dtype=float)
    A, C = a/2, c/2
    J0 = np.zeros_like(x0)
    J1 = np.zeros_like(x0)
    for lower, upper in [(-A, -C), (C, A)]:
        left = np.maximum(x0, lower)
        active = left < upper
        j0 = 0.5*(upper**2 - left**2) - x0*(upper - left)
        j1 = (upper**3 - left**3)/3 - 0.5*x0*(upper**2 - left**2)
        J0 += np.where(active, j0, 0.0)
        J1 += np.where(active, j1, 0.0)
    return J0, J1

def solve_theta_hollow_linear(thetas, V, a, c, ks, maxiter=70):
    thetas = np.atleast_1d(np.asarray(thetas, dtype=float))
    st, ct = np.sin(thetas), np.cos(thetas)
    zero_mask = np.abs(st) < 1e-12
    st_safe = np.where(zero_mask, 1.0, st)
    contact_width = a - c

    # Nullstelle x0 der linearen Setzung s=sin(theta)*(x-x0).
    lo = -a/2 - 2*V/(ks*st_safe*contact_width)
    hi = np.full_like(thetas, a/2)
    for _ in range(maxiter):
        mid = 0.5*(lo + hi)
        J0, _ = _hollow_active_integrals(mid, a, c)
        resid = ks*st_safe*J0 - V
        lo = np.where(resid > 0, mid, lo)
        hi = np.where(resid > 0, hi, mid)
    x0 = 0.5*(lo + hi)
    _, J1 = _hollow_active_integrals(x0, a, c)
    s0 = -x0*st_safe
    M = ks*st_safe*ct*J1
    s0_zero = V/(ks*contact_width)
    s0 = np.where(zero_mask, s0_zero, s0)
    M = np.where(zero_mask, 0.0, M)
    return s0, M


In [ ]:
def plot_hollow_foundation_characteristics(a=4.0):
    r = np.linspace(0.0, 0.9, 400)
    e1_to_a = (1 + r + r**2)/6
    e2_to_a = (1 + r + r**2)/(3*(1 + r))
    kscrit_ratio = 1/(1 - r**3)

    fig, ax = plt.subplots(ncols=2, figsize=(15, 5.8))
    ax[0].plot(r, e1_to_a, color='seagreen', lw=2.5, label=r'1. Kernweite $e_1/a$')
    ax[0].plot(r, e2_to_a, color='darkorange', lw=2.5, label=r'2. Kernweite $e_2/a$')
    ax[0].axhline(0.5, color='darkred', lw=2, ls='--', label=r'Kippgrenze $a/2$')
    ax[0].set_xlabel(r'Kernbreite $c/a$')
    ax[0].set_ylabel(r'bezogene Exzentrizität $e/a$')
    ax[0].set_xlim(0, 0.9)
    ax[0].set_ylim(0, 0.52)
    ax[0].legend(fontsize=11)
    ax[0].set_title('Kernweiten des durchbrochenen Fundaments')

    ax[1].plot(r, kscrit_ratio, color='black', lw=2.5)
    ax[1].axhline(1.0, color='grey', lw=1, ls=':')
    ax[1].set_xlabel(r'Kernbreite $c/a$')
    ax[1].set_ylabel(r'$k_{s,\mathrm{krit}}(c)/k_{s,\mathrm{krit}}(0)$')
    ax[1].set_xlim(0, 0.9)
    ax[1].set_ylim(bottom=0.9)
    ax[1].set_title('Erforderliche Bettungssteifigkeit')
    fig.tight_layout()
    plt.show();

In [ ]:
def plot_stability_map(a=4.0, H=20.0, c_ratios=(0.0,)):
    zG0 = H/2
    V = 25.0*a*H
    ks_crit_full = 12*V*zG0/a**3
    theta_rigid = theta_crit_rigid(a, zG0)
    thetas = np.linspace(1e-4, theta_rigid*0.999, 300)
    k_ratios = np.logspace(np.log10(0.2), 1, 200)
    TH, KR = np.meshgrid(np.degrees(thetas), k_ratios)

    c_ratios = tuple(c_ratios)
    if any((r < 0 or r >= 1) for r in c_ratios):
        raise ValueError('Für alle Werte muss 0 <= c/a < 1 gelten.')
    fig, axes = plt.subplots(ncols=len(c_ratios), figsize=(6*len(c_ratios), 6.5),
                             sharex=True, sharey=True, squeeze=False,
                             constrained_layout=True)
    axes = axes.ravel()
    G_full = np.zeros((len(k_ratios), len(thetas)))
    for i, kr in enumerate(k_ratios):
        ks = kr*ks_crit_full
        _, Mf_full = solve_theta_hollow_linear(thetas, V, a, 0.0, ks)
        G_full[i, :] = Mf_full - M_dest(thetas, V, zG0)

    cset = None
    for ax, r in zip(axes, c_ratios):
        c = r*a
        G = np.zeros((len(k_ratios), len(thetas)))
        for i, kr in enumerate(k_ratios):
            ks = kr*ks_crit_full
            _, Mf = solve_theta_hollow_linear(thetas, V, a, c, ks)
            G[i, :] = Mf - M_dest(thetas, V, zG0)
        cset = ax.contourf(TH, KR, np.sign(G), cmap=cm.coolwarm_r,
                           levels=[-1, 0, 1], alpha=0.35)
        ax.contour(TH, KR, G, levels=[0], colors='black', linewidths=2)
        ax.plot([], [], color='black', lw=2,
                label=fr'Stabilitätsgrenze $c/a={r:.1f}$')
        if not np.isclose(r, 0.0):
            ax.contour(TH, KR, G_full, levels=[0], colors='black',
                       linewidths=2, linestyles='--')
            ax.plot([], [], color='black', lw=2, ls='--',
                    label=r'Referenz $c/a=0$')
        local_limit = 1/(1 - r**3)
        ax.axhline(local_limit, color='grey', ls='--',
                   label=fr'lokale Grenze $={local_limit:.2f}\,k_{{s,\mathrm{{krit}}}}(0)$')
        ax.set_yscale('log')
        ax.set_xlabel(r'$\theta$ / deg')
        ax.set_title(fr'$c/a={r:.1f}$')
        ax.legend(loc='upper right', fontsize=9)
    axes[0].set_ylabel(r'$k_s\,/\,k_{s,\mathrm{krit}}(0)$')
    fig.suptitle(r'Momentenresiduum $G=M_\mathrm{found}-M_\mathrm{dest}$')
    fig.colorbar(cset, ax=axes.tolist(), label=r'sign $G$: $-$ kippend, $+$ rückstellend',
                 shrink=0.88)
    plt.show();